# Can we predict a warranty claim within a year?
Target: `claim_within_year`. Following `checklist.md`.

**Data facts that drive the design**
- `device_serial` is an identifier -> dropped (checklist 7). Every serial is unique, so each entity is one row.
- There is no date column, so there is no later period to forecast and no basis for a time split; a random stratified split is used. (If rows are in fact in time order, this would be optimistic.)
- All remaining columns (channel, device age at sale, price, reported daily use at registration) are recorded at or before the sale, so they are known at prediction time.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix
SEED=0
df=pd.read_csv('data.csv')
print(df.shape, df.device_serial.is_unique)
print(df.describe().T)
print(df.channel.value_counts()); print('claim rate', df.claim_within_year.mean())

## 1. Hold out the test set first
Stratified 20% random hold-out (entities are unique, no time column). It is not touched until step 6.

In [ ]:
y=df.claim_within_year; X=df.drop(columns=['claim_within_year','device_serial'])
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
print(len(X_tr),len(X_te),y_tr.mean(),y_te.mean())

## 2-4. Candidates (named before any scoring) and CV scheme
Candidates:
1. `baseline` - predicts the training prior (trivial).
2. `logreg` - logistic regression on scaled numerics + one-hot channel; C in {0.01, 0.1, 1, 10}.
3. `hgb` - histogram gradient boosting; learning_rate {0.03, 0.1}, max_depth {2, 3}.
4. `rf` - random forest; min_samples_leaf {5, 20, 50}.

Selection metric: average precision (PR-AUC), since only ~13% of rows are positive. Threshold: chosen on out-of-fold predictions to maximise F1. CV: 5-fold stratified on training rows only (rows are unique entities). Preprocessing lives inside the pipeline so it is fit per fold.

In [ ]:
num=['device_age_years','price_eur','daily_hours']; cat=['channel']
def pre(scale):
    return ColumnTransformer([('n',StandardScaler() if scale else 'passthrough',num),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
cands={
 'baseline':(Pipeline([('p',pre(False)),('m',DummyClassifier(strategy='prior'))]),{}),
 'logreg':(Pipeline([('p',pre(True)),('m',LogisticRegression(max_iter=1000))]),{'m__C':[0.01,0.1,1,10]}),
 'hgb':(Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(random_state=SEED))]),{'m__learning_rate':[0.03,0.1],'m__max_depth':[2,3]}),
 'rf':(Pipeline([('p',pre(False)),('m',RandomForestClassifier(n_estimators=300,random_state=SEED))]),{'m__min_samples_leaf':[5,20,50]}),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
fits={}
for k,(pipe,grid) in cands.items():
    g=GridSearchCV(pipe,grid or {'m__strategy':['prior']},scoring='average_precision',cv=cv,n_jobs=1).fit(X_tr,y_tr)
    fits[k]=g
    r=g.cv_results_; i=g.best_index_
    print(f'{k:9s} CV AP {r["mean_test_score"][i]:.4f} +/- {r["std_test_score"][i]:.4f}  {g.best_params_ if grid else ""}')

## 5. Choose model and threshold on CV only

In [ ]:
ap={k:g.best_score_ for k,g in fits.items()}
best=max(ap,key=ap.get)
oof=cross_val_predict(fits[best].best_estimator_,X_tr,y_tr,cv=cv,method='predict_proba')[:,1]
ths=np.linspace(0.02,0.6,59)
f1s=[f1_score(y_tr,oof>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('CV AP of baseline (= prevalence):',round(ap['baseline'],4))
print('chosen:',best,fits[best].best_params_,'threshold',round(thr,3),'OOF F1',round(max(f1s),3))
print('OOF ROC AUC',round(roc_auc_score(y_tr,oof),4))

**Decision (frozen before test scoring):** the model with the highest cross-validated average precision, as printed above, with its CV-tuned hyperparameters and the F1-maximising out-of-fold threshold. Nothing below changes these. If the winner's CV AP is not clearly above the baseline's, the honest answer is that the features carry little signal.

## 6. Refit on all training rows, score the test set once

In [ ]:
final=fits[best].best_estimator_.fit(X_tr,y_tr)
p=final.predict_proba(X_te)[:,1]; pred=p>=thr
print('Test average precision (headline):',round(average_precision_score(y_te,p),4),'| prevalence',round(y_te.mean(),4))
print('Test ROC AUC:',round(roc_auc_score(y_te,p),4))
print('At threshold %.3f: precision %.3f recall %.3f F1 %.3f'%(thr,precision_score(y_te,pred),recall_score(y_te,pred),f1_score(y_te,pred)))
print(confusion_matrix(y_te,pred))
rng=np.random.default_rng(SEED); yt=y_te.to_numpy(); bs=[]
for _ in range(1000):
    i=rng.integers(0,len(yt),len(yt)); bs.append(average_precision_score(yt[i],p[i]))
print('95%% bootstrap CI for AP: %.3f-%.3f'%tuple(np.percentile(bs,[2.5,97.5])))

## Conclusion
Read the headline test average precision against the prevalence (the baseline's AP). Interpret only the gap, and use the CI to judge whether it is real. Caveats: no dates, so no out-of-time validation; a single hold-out of 1,400 rows (~176 positives) gives a fairly wide interval.